In [ ]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Import TensorFlow and Keras components
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from scikeras.wrappers import KerasClassifier # For GridSearchCV compatibility

# Function to extract MFCCs from audio files
def extract_mfccs(file_path):
    try:
        y, sr = librosa.load(file_path, duration=1)  # Load 1-second audio
        # Ensure n_mfcc is consistent with the model's input expectation
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=30)
        return mfccs.T  # Transpose to get (timesteps, n_mfcc) for CNN
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Data Loading and Feature Extraction
adult_folder = "/home/feliciano/RE_001503_Acoustic_TAC_project2/fish1"
juvenile_folder = "/home/feliciano/RE_001503_Acoustic_TAC_project2/fish2"

data = []
labels = []

for filename in os.listdir(adult_folder):
    file_path = os.path.join(adult_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("adult")

for filename in os.listdir(juvenile_folder):
    file_path = os.path.join(juvenile_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("juvenile")

if not data:
    print("No audio files found in the specified folders. Please check your paths and file types.")
    exit()

# Pad sequences to ensure uniform length for CNN input
# Find the maximum sequence length among all MFCCs
max_pad_len = max(mfcc.shape[0] for mfcc in data)

# Pad all MFCC sequences to the max_pad_len
padded_data = []
for mfcc in data:
    # Pad or truncate MFCCs to max_pad_len. Using 'reflect' padding.
    if mfcc.shape[0] < max_pad_len:
        pad_width = ((0, max_pad_len - mfcc.shape[0]), (0, 0))
        padded_mfcc = np.pad(mfcc, pad_width=pad_width, mode='constant', constant_values=0)
    else: # If mfcc.shape[0] > max_pad_len, truncate
        padded_mfcc = mfcc[:max_pad_len, :]
    padded_data.append(padded_mfcc)

X = np.array(padded_data)
y = np.array(labels)

# Label Encoding
le = LabelEncoder()
y = le.fit_transform(y)
class_names = le.classes_ # Store class names for plotting

# Data Scaling (Per-feature scaling across all timesteps)
# Reshape X for StandardScaler: (n_samples * n_timesteps, n_mfcc)
original_shape = X.shape
X_reshaped_for_scaler = X.reshape(-1, original_shape[2])

scaler = StandardScaler()
X_scaled_reshaped = scaler.fit_transform(X_reshaped_for_scaler)

# Reshape back to original 3D shape: (n_samples, n_timesteps, n_mfcc)
X = X_scaled_reshaped.reshape(original_shape)

# Train-Test Split
# Ensure the input to CNN is (samples, timesteps, features)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---
## 1D Convolutional Neural Network (1D-CNN) Model with Hyperparameter Optimization

# Define a function to create the 1D-CNN model
def create_cnn_model(filters=64, kernel_size=5, activation='relu', dropout_rate=0.5, learning_rate=0.001, input_shape=None):
    model = Sequential([
        Conv1D(filters=filters, kernel_size=kernel_size, activation=activation, input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Dropout(dropout_rate),
        Flatten(),
        Dense(64, activation='relu'), # A small dense layer before the output
        Dense(1, activation='sigmoid') # Binary classification
    ])
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Create a KerasClassifier wrapper for GridSearchCV
# Pass the input_shape to the build_fn
# Note: X_train.shape[1:] gives (timesteps, features)
cnn_model = KerasClassifier(
    model=create_cnn_model,
    input_shape=X_train.shape[1:], # Pass the correct input shape
    verbose=0 # Suppress Keras output during GridSearchCV
)

# Define the parameter grid for GridSearchCV for 1D-CNN
param_grid = {
    'model__filters': [32, 64, 128],  # Number of filters in Conv1D layer
    'model__kernel_size': [3, 5, 7],  # Kernel size for Conv1D layer
    'model__activation': ['relu', 'tanh'], # Activation for Conv1D
    'model__dropout_rate': [0.2, 0.4, 0.5], # Dropout rate
    'model__learning_rate': [0.001, 0.0005], # Learning rate for Adam optimizer
    'batch_size': [16, 32], # Batch size for training
    'epochs': [20, 30] # Number of epochs for training
}

# Setup GridSearchCV
# Note: Using 'accuracy' as scoring
grid_search = GridSearchCV(cnn_model, param_grid, cv=3, n_jobs=-1, verbose=2, scoring='accuracy')

print("Starting GridSearchCV for 1D-CNN optimization...")
start_time = time.time()
grid_search.fit(X_train, y_train) # Fit the GridSearchCV object
end_time = time.time()
optimization_time = (end_time - start_time) * 1000 # milliseconds

print(f"1D-CNN Optimization Time (ms): {optimization_time:.4f}")

# Get the best model from GridSearchCV
best_cnn_model = grid_search.best_estimator_

print("\nBest 1D-CNN Parameters found:")
print(grid_search.best_params_)
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# ---
## Model Evaluation with Best 1D-CNN Model

y_pred = best_cnn_model.predict(X_test)
y_pred_proba = best_cnn_model.predict_proba(X_test)[:, 1] # Probability of the positive class (juvenile)

# Determine the index of the 'juvenile' class after encoding
juvenile_class_index = np.where(le.classes_ == 'juvenile')[0][0] # This will be 1 if 'juvenile' is second alphabetically

# Calculate Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=juvenile_class_index, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=juvenile_class_index, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=juvenile_class_index, zero_division=0)

# Check if there are both positive and negative samples in y_test for AUC calculation
if len(np.unique(y_test)) > 1:
    auc_roc = roc_auc_score(y_test, y_pred_proba)
else:
    auc_roc = float('nan') # Not applicable if only one class is present in y_test

# Print Results
print("\n1D-CNN Model Performance (Optimized):")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision (Juvenile): {precision:.4f}")
print(f"  Recall (Juvenile): {recall:.4f}")
print(f"  F1-Score (Juvenile): {f1:.4f}")
print(f"  AUC-ROC: {auc_roc:.4f}")
print(f"  Inference Time (ms): {(time.time() - end_time) * 1000:.4f}") # Re-calculate for test set prediction

# ---
## Confusion Matrix Implementation

# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("\n---")
print("### Original Confusion Matrix")
print("---")
print(cm)

# Plotting the original Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Counts) for Optimized 1D-CNN')
plt.show()

# --- Normalizing the Confusion Matrix ---
# Divide each row by the sum of the row
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

print("\n---")
print("### Normalized Confusion Matrix (Rows sum to 1.00)")
print("---")
print(cm_normalized)

# Plotting the Normalized Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', # fmt='.2f' for 2 decimal places
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Normalized Confusion Matrix (Optimized 1D-CNN on fish1 vs fish2)')
plt.show()

print(f"\nClass mapping (0 to N): {list(le.classes_)}")

2025-05-29 16:34:50.518410: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748532891.085932   34619 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748532891.247289   34619 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748532892.568677   34619 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748532892.568714   34619 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748532892.568717   34619 computation_placer.cc:177] computation placer alr

Starting GridSearchCV for 1D-CNN optimization...
Fitting 3 folds for each of 432 candidates, totalling 1296 fits


2025-05-29 16:35:06.567683: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748532906.591258   38751 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748532906.599595   38751 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748532906.617594   38751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748532906.617786   38751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748532906.617868   38751 computation_placer.cc:177] computation placer alr

[CV] END ........C=0.1, degree=2, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ............C=0.1, degree=2, gamma=auto, kernel=rbf; total time=   0.0s
[CV] END ............C=0.1, degree=2, gamma=auto, kernel=rbf; total time=   0.0s
[CV] END ............C=0.1, degree=2, gamma=auto, kernel=rbf; total time=   0.0s
[CV] END ............C=0.1, degree=2, gamma=auto, kernel=rbf; total time=   0.0s
[CV] END ............C=0.1, degree=2, gamma=auto, kernel=rbf; total time=   0.0s
[CV] END ...........C=0.1, degree=2, gamma=auto, kernel=poly; total time=   0.0s
[CV] END ...........C=0.1, degree=2, gamma=auto, kernel=poly; total time=   0.0s
[CV] END ...........C=0.1, degree=2, gamma=auto, kernel=poly; total time=   0.0s
[CV] END ..........C=0.1, degree=2, gamma=0.1, kernel=linear; total time=   0.0s
[CV] END ..........C=0.1, degree=2, gamma=0.1, kernel=linear; total time=   0.0s
[CV] END .............C=0.1, degree=2, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .............C=0.1,

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
2025-05-29 16:35:

...

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model i

[CV] END batch_size=32, epochs=20, model__activation=relu, model__dropout_rate=0.4, model__filters=128, model__kernel_size=5, model__learning_rate=0.001; total time=   4.2s
[CV] END batch_size=32, epochs=20, model__activation=relu, model__dropout_rate=0.5, model__filters=64, model__kernel_size=5, model__learning_rate=0.0005; total time=   4.0s
[CV] END batch_size=32, epochs=20, model__activation=relu, model__dropout_rate=0.5, model__filters=128, model__kernel_size=5, model__learning_rate=0.0005; total time=   4.6s
[CV] END batch_size=32, epochs=20, model__activation=tanh, model__dropout_rate=0.2, model__filters=32, model__kernel_size=7, model__learning_rate=0.001; total time=   3.9s
[CV] END batch_size=32, epochs=20, model__activation=tanh, model__dropout_rate=0.2, model__filters=64, model__kernel_size=7, model__learning_rate=0.001; total time=   4.2s
[CV] END batch_size=32, epochs=20, model__activation=tanh, model__dropout_rate=0.2, model__filters=128, model__kernel_size=7, model__lea

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model i

[CV] END batch_size=32, epochs=20, model__activation=tanh, model__dropout_rate=0.4, model__filters=32, model__kernel_size=7, model__learning_rate=0.0005; total time=   4.6s
[CV] END batch_size=32, epochs=20, model__activation=tanh, model__dropout_rate=0.5, model__filters=32, model__kernel_size=7, model__learning_rate=0.001; total time=   3.3s
[CV] END batch_size=32, epochs=20, model__activation=tanh, model__dropout_rate=0.5, model__filters=64, model__kernel_size=5, model__learning_rate=0.0005; total time=   3.5s
[CV] END batch_size=32, epochs=20, model__activation=tanh, model__dropout_rate=0.5, model__filters=128, model__kernel_size=5, model__learning_rate=0.001; total time=   4.5s
[CV] END batch_size=32, epochs=30, model__activation=relu, model__dropout_rate=0.2, model__filters=32, model__kernel_size=3, model__learning_rate=0.0005; total time=   4.9s
[CV] END batch_size=32, epochs=30, model__activation=relu, model__dropout_rate=0.2, model__filters=64, model__kernel_size=5, model__lear

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model i

In [4]:
import os
#print(dir(os))
print(os.listdir)

<built-in function listdir>
